# Tugas 3 · Klaster Material Multimodal

Notebook mandiri untuk eksplorasi, pembelajaran embedding, pengelompokan, dan reproduksi bagian laporan Tugas 3. Pipeline mengikuti versi ringkas dari notebook gabungan. Jalankan Sel 1–4, panggil Sel 5 untuk eksekusi, lalu gunakan Sel 6 untuk mengekspor hasil.

## Sel 1 · Masukan dan keluaran

Colab membaca `clustering.npz` dan metadata sampel dari folder data UTS di Drive. `kamus_data.md` menyertai dataset sebagai dokumentasi. Gunakan jalur aktual pada `DATA` jika susunan Drive berbeda. Keluaran disimpan di runtime pada `HASIL`. Untuk Jupyter lokal, jalur dapat diatur lewat `UTS_DATA_DIR` dan `UTS_HASIL_TUGAS3`.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    if "DATA" not in globals() and not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    default_data = "/content/drive/MyDrive/UTS_DL/data_uts_dl"
    default_hasil = "/content/hasil_tugas3"
else:
    default_data = "data_uts_dl"
    default_hasil = "hasil_tugas3"

DATA = Path(globals().get("DATA", os.environ.get("UTS_DATA_DIR", default_data)))
HASIL = Path(globals().get("HASIL", os.environ.get("UTS_HASIL_TUGAS3", default_hasil)))
HASIL.mkdir(parents=True, exist_ok=True)

berkas = ["clustering.npz", "clustering_meta.csv", "kamus_data.md"]
kurang = [nama for nama in berkas if not (DATA / nama).is_file()]
if kurang:
    raise FileNotFoundError(f"Periksa folder {DATA}: " + ", ".join(kurang))

print("Data:", DATA)
print("Keluaran:", HASIL)

## Sel 2 · Dependensi dan verifikasi metadata

PyTorch menjalankan kedua autoencoder dan pembaruan DEC. SciPy menghitung deskriptor Fourier dan menyaring spektrum. scikit-learn memasang K-means, menghitung metrik internal, dan membuat t-SNE. Fungsi pembaca mengurutkan metadata berdasarkan indeks resmi dan memeriksa kecocokannya dengan array. CPU empat thread dipakai sebagai konfigurasi referensi.

In [ ]:
%pip -q install reportlab pypdf

import json
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import torch

from scipy import fft, ndimage
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_score
from threadpoolctl import threadpool_limits
from torch import nn
from torch.nn import functional as F

DEVICE = torch.device("cpu")
torch.set_num_threads(4)
print("Perangkat:", DEVICE)


def seed_semua(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def tensor(x):
    return torch.as_tensor(x, device=DEVICE)


def baca_npz(nama, modalitas, id_kolom):
    meta = pd.read_csv(DATA / f"{nama}_meta.csv")
    meta = meta.sort_values("indeks").reset_index(drop=True)
    with np.load(DATA / f"{nama}.npz", allow_pickle=False) as arsip:
        gambar, kedua = arsip["gambar"], arsip[modalitas]
    assert len(meta) == len(gambar) == len(kedua)
    assert meta[id_kolom].is_unique
    np.testing.assert_array_equal(meta.indeks, np.arange(len(meta)))
    return meta, gambar, kedua


def simpan_grafik(fig, nama):
    fig.savefig(HASIL / nama, dpi=140, bbox_inches="tight")
    plt.close(fig)

## Sel 3 · Pipeline klaster multimodal

**Eksplorasi.** Perbandingan menurut mikroskop mencakup kecerahan, orientasi, proksi skala tekstur, baseline, spike, dan posisi puncak spektrum.

**Representasi.** Dua denoising autoencoder dilatih dari awal. Cabang citra memakai ringkasan Fourier, sedangkan cabang spektrum menerima profil yang dibersihkan. Noise dan pergeseran bin dipakai sebagai augmentasi. Embedding kemudian distandardisasi dan digabung.

**Pengelompokan.** K-means menguji kandidat jumlah klaster dan membandingkan model satu modalitas dengan fusi. Jumlah dipilih dari silhouette dan kestabilan antarresampling. DEC memperbarui encoder dan pusat klaster. t-SNE, margin penugasan, dan prototipe dipakai untuk meninjau pasangan yang sulit dipisahkan.

ARI di notebook ini mengukur kesepakatan antarpartisi. Skor penilaian memakai label referensi yang disimpan pengajar.

In [ ]:
def tekstur(gambar):
    # Ringkasan spektrum Fourier yang kurang sensitif pada arah gambar.
    f = np.fft.fftfreq(64) * 64
    yy, xx = np.meshgrid(f, f, indexing="ij")
    radius = np.hypot(xx, yy).ravel()
    sudut = np.arctan2(yy, xx).ravel()
    bins = np.arange(1, 29)
    cincin = ((radius[:, None] >= bins) & (radius[:, None] < bins+1)).astype(np.float32)
    harmonik = np.column_stack([
        np.cos(2*sudut), np.sin(2*sudut), np.cos(4*sudut), np.sin(4*sudut),
    ]).astype(np.float32)
    hann = np.outer(np.hanning(64), np.hanning(64)).astype(np.float32)

    fitur, arah, frekuensi = [], [], []
    for awal in range(0, len(gambar), 256):
        x = gambar[awal:awal+256].astype(np.float32)
        x -= x.mean((1, 2), keepdims=True)
        x /= x.std((1, 2), keepdims=True).clip(1)
        daya = np.abs(fft.fft2(x * hann)).reshape(len(x), -1)**2
        daya[:, 0] = 0
        daya /= daya.sum(1, keepdims=True).clip(1e-7)
        radial = ndimage.gaussian_filter1d(daya @ cincin, 1.1, axis=1)
        c2, s2, c4, s4 = (daya @ harmonik).T
        fitur.append(np.column_stack([np.log1p(60*radial), np.hypot(c2, s2), np.hypot(c4, s4)]))
        arah.append((np.rad2deg(np.arctan2(s2, c2))/2) % 180)
        frekuensi.append(radial.argmax(1) + 1)
    return np.vstack(fitur), np.concatenate(arah), np.concatenate(frekuensi)


class AE(nn.Module):
    def __init__(self, dim, laten):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(dim, 96), nn.ReLU(), nn.Linear(96, laten))
        self.dec = nn.Sequential(nn.Linear(laten, 96), nn.ReLU(), nn.Linear(96, dim))


def tugas3():
    seed_semua(17)
    rng = np.random.default_rng(17)
    m, gambar, spektrum = baca_npz("clustering", "spektrum", "id_sampel")
    assert np.isfinite(spektrum).all()

    g, arah, frekuensi = tekstur(gambar)
    mentah = spektrum.astype(np.float32)
    med = ndimage.median_filter(mentah, size=(1, 3), mode="nearest")
    spike = np.abs(mentah - med) > .55
    s = np.where(spike, med, mentah)
    baseline = ndimage.gaussian_filter1d(ndimage.grey_opening(s, size=(1, 23)), 4, axis=1)
    s = ndimage.gaussian_filter1d((s-baseline).clip(0), 1.15, axis=1)
    s /= np.linalg.norm(s, axis=1, keepdims=True).clip(1e-6)

    terang = gambar.mean((1, 2))
    distorsi = pd.DataFrame({
        "mikroskop": m.mikroskop, "terang": terang, "frekuensi": frekuensi,
        "baseline": np.ptp(baseline, axis=1), "spike": spike.any(1), "puncak": s.argmax(1),
    }).groupby("mikroskop", as_index=False).agg(
        n=("terang", "size"), terang=("terang", "mean"), frekuensi=("frekuensi", "median"),
        baseline=("baseline", "mean"), spike=("spike", "mean"), puncak=("puncak", "median"),
    )

    fig, ax = plt.subplots(2, 4, figsize=(11, 5), layout="constrained")
    for j, mik in enumerate(sorted(m.mikroskop.unique())):
        ids = np.flatnonzero(m.mikroskop == mik)
        dengan_spike = ids[spike[ids].any(1)]
        i = dengan_spike[0] if len(dengan_spike) else ids[0]
        ax[j, 0].imshow(gambar[i], cmap="gray", vmin=50, vmax=205)
        ax[j, 0].set_title(mik)
        ax[j, 0].axis("off")
        ax[j, 1].plot(mentah[i], label="Mentah")
        ax[j, 1].plot(baseline[i], label="Baseline")
        ax[j, 1].plot(s[i], label="Bersih")
        ax[j, 1].legend(fontsize=7)
        ax[j, 2].hist(arah[ids], bins=18)
        ax[j, 2].set_title("Orientasi Fourier")
        ax[j, 3].hist(frekuensi[ids], bins=np.arange(1, 30))
        ax[j, 3].set_title("Frekuensi dominan")
    simpan_grafik(fig, "t3_distorsi.png")

    tg = tensor(((g-g.mean(0))/g.std(0).clip(.025)).astype(np.float32))
    ts = tensor((s/max(float(s.std()), .001)).astype(np.float32))
    ae_g, ae_s = AE(tg.shape[1], 10).to(DEVICE), AE(ts.shape[1], 14).to(DEVICE)
    parameter = [*ae_g.parameters(), *ae_s.parameters()]
    opt = torch.optim.Adam(parameter, lr=.002)

    acak = torch.randperm(len(tg), device=DEVICE)
    dev, fit = acak[:len(tg)//10], acak[len(tg)//10:]

    def ganggu(x, geser):
        offset = int(rng.integers(-geser, geser+1))
        y = x.roll(offset, dims=1)
        if offset > 0:
            y[:, :offset] = 0
        if offset < 0:
            y[:, offset:] = 0
        return y

    rekonstruksi = []
    for epoch in range(1, 19):
        for i in fit[torch.randperm(len(fit), device=DEVICE)].split(384):
            # Pergeseran radial kecil menjadi proksi perubahan skala.
            g_aug = torch.cat([ganggu(tg[i, :28], 1), tg[i, 28:]], dim=1)
            s_aug = ganggu(ts[i], 3)
            g_aug = g_aug + .07 * torch.randn_like(g_aug)
            s_aug = s_aug + .07 * torch.randn_like(s_aug)
            loss = F.mse_loss(ae_g.dec(ae_g.enc(g_aug)), tg[i])
            loss = loss + F.mse_loss(ae_s.dec(ae_s.enc(s_aug)), ts[i])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        with torch.no_grad():
            mse_g = F.mse_loss(ae_g.dec(ae_g.enc(tg[dev])), tg[dev]).item()
            mse_s = F.mse_loss(ae_s.dec(ae_s.enc(ts[dev])), ts[dev]).item()
        rekonstruksi.append((epoch, mse_g, mse_s))
        print(f"T3 AE | epoch {epoch}/18 | MSE {mse_g:.3f}, {mse_s:.3f}")

    with torch.no_grad():
        zg, zs = ae_g.enc(tg), ae_s.enc(ts)
        mg, ms = zg.mean(0), zs.mean(0)
        sg, ss = zg.std(0).clamp_min(.02), zs.std(0).clamp_min(.02)

    def gabung(a, b):
        a = (a-mg)/sg / math.sqrt(a.shape[1])
        b = (b-ms)/ss / math.sqrt(b.shape[1])
        return a, b, torch.cat([.70*a, b], dim=1)

    with torch.no_grad():
        gi, si, z = [t.cpu().numpy() for t in gabung(zg, zs)]

    urut = rng.permutation(len(z))
    nilai, kandidat = urut[:1500], urut[1500:]
    a, b = [rng.choice(kandidat, 6000, replace=False) for _ in range(2)]
    cari = []
    for k in range(2, 13):
        k1 = KMeans(k, n_init=5, random_state=17).fit(z[a])
        k2 = KMeans(k, n_init=5, random_state=29).fit(z[b])
        p1, p2 = k1.predict(z[nilai]), k2.predict(z[nilai])
        sil = silhouette_score(z[nilai], p1)
        stabil = adjusted_rand_score(p1, p2)
        cari.append((k, sil, stabil, sil+.10*stabil))
    pilihan = pd.DataFrame(cari, columns=["k", "silhouette", "stabilitas", "gabungan"])
    k = int(pilihan.loc[pilihan.gabungan.idxmax(), "k"])
    pilihan.to_csv(HASIL / "t3_pilihan_k.csv", index=False)

    ruang = {"Citra": gi, "Spektrum": si, "Fusi": z}
    km = {nama: KMeans(k, n_init=10, random_state=17).fit(x) for nama, x in ruang.items()}
    awal = km["Fusi"].labels_
    idx = rng.choice(len(z), 2000, replace=False)
    ablasi = pd.DataFrame([
        {
            "modalitas": nama,
            "silhouette sendiri": silhouette_score(ruang[nama][idx], net.labels_[idx]),
            "silhouette ruang fusi": silhouette_score(z[idx], net.labels_[idx]),
            "ARI vs fusi": adjusted_rand_score(awal, net.labels_),
        }
        for nama, net in km.items()
    ])

    pusat = nn.Parameter(tensor(km["Fusi"].cluster_centers_.copy()))
    opt = torch.optim.Adam([*parameter, pusat], lr=.00025)

    def probabilitas(z):
        q = (1 + (z[:, None] - pusat[None]).square().sum(2)).reciprocal()
        return q / q.sum(1, keepdim=True)

    for epoch in range(1, 9):
        with torch.no_grad():
            q = probabilitas(gabung(ae_g.enc(tg), ae_s.enc(ts))[2])
            target = q.square() / q.sum(0, keepdim=True)
            target = target / target.sum(1, keepdim=True)
        for i in torch.randperm(len(tg), device=DEVICE).split(384):
            a, b = ae_g.enc(tg[i]), ae_s.enc(ts[i])
            pred = probabilitas(gabung(a, b)[2])
            kl = F.kl_div(pred.log(), target[i], reduction="batchmean")
            recon = F.mse_loss(ae_g.dec(a), tg[i]) + F.mse_loss(ae_s.dec(b), ts[i])
            loss = kl + .30*recon
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        print(f"T3 DEC | epoch {epoch}/8 | loss batch terakhir {loss.item():.4f}")

    with torch.no_grad():
        laten_dec = gabung(ae_g.enc(tg), ae_s.enc(ts))[2]
        q = probabilitas(laten_dec).cpu().numpy()
        laten_dec = laten_dec.cpu().numpy()

    label_dec = q.argmax(1)
    jumlah_dec = np.unique(label_dec).size
    if jumlah_dec == k:
        label, laten, metode = label_dec, laten_dec, "DEC"
    else:
        label, laten, metode = awal, z, "K-means pada embedding autoencoder"

    margin = np.diff(np.sort(q, axis=1)[:, -2:], axis=1).ravel()
    batas = np.quantile(margin, .15)
    pasangan = np.sort(np.argsort(q, axis=1)[:, -2:], axis=1)
    unik, hitung = np.unique(pasangan[margin <= batas], axis=0, return_counts=True)
    sulit = unik[hitung.argmax()]
    ari = adjusted_rand_score(awal, label_dec)
    sil_awal = silhouette_score(z[idx], awal[idx])
    sil_final = silhouette_score(laten[idx], label[idx])
    sil_dec_awal = (
        f"{silhouette_score(z[idx], label_dec[idx]):.3f}"
        if np.unique(label_dec[idx]).size > 1 else "tidak terdefinisi"
    )

    pd.DataFrame({"id_sampel": m.id_sampel, "cluster": label}).to_csv(HASIL / "cluster.csv", index=False)
    pd.DataFrame({
        "id_sampel": m.id_sampel, "KMeans": awal, "DEC": label_dec, "margin_DEC": margin,
    }).to_csv(HASIL / "t3_pembanding.csv", index=False)

    peta = TSNE(n_components=2, perplexity=30, init="pca", random_state=17).fit_transform(laten[idx])
    fig, ax = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    for kl in np.unique(label):
        i = label[idx] == kl
        ax[0].scatter(peta[i, 0], peta[i, 1], s=6, label=str(kl))
    ax[0].set_title(f"t-SNE {metode}")
    ax[0].legend(title="Klaster", fontsize=7)
    sc = ax[1].scatter(peta[:, 0], peta[:, 1], s=6, c=margin[idx], cmap="viridis")
    ax[1].set_title("Margin dua penugasan DEC teratas")
    fig.colorbar(sc, ax=ax[1])
    simpan_grafik(fig, "t3_tsne.png")

    fig, ax = plt.subplots(1, 2, figsize=(9, 3), layout="constrained")
    puncak = []
    for a, kl in zip(ax, sulit):
        i = np.flatnonzero(label_dec == kl)
        if len(i):
            tengah = np.median(s[i], axis=0)
            puncak.append(int(tengah.argmax()))
            j = i[np.argmin(((s[i]-tengah)**2).sum(1))]
            a.plot(tengah)
            foto = a.inset_axes([.66, .50, .32, .45])
            foto.imshow(gambar[j], cmap="gray")
            foto.axis("off")
        a.set_title(f"Prototipe DEC {kl}")
    simpan_grafik(fig, "t3_prototipe.png")

    kurva = pd.DataFrame(rekonstruksi, columns=["epoch", "MSE citra", "MSE spektrum"])
    kurva.to_csv(HASIL / "t3_autoencoder.csv", index=False)

    return k, [
        (
            "Tugas 3 | Distorsi dan embedding",
            [
                "Kecerahan, orientasi Fourier, dan frekuensi dominan dibandingkan antarmikroskop. Frekuensi menjadi proksi skala. Komposisi sampel juga dapat memengaruhi perbedaan antarmikroskop.",
                "Median lokal menekan spike. Morphological opening memperkirakan baseline. Puncak dominan dibandingkan antarmikroskop sebagai indikasi variasi posisi spektral.",
                "Autoencoder citra dan spektrum merekonstruksi fitur bersih dari input bergeser dan diberi noise. Pergeseran radial kecil menyimulasikan perubahan skala, sedangkan pergeseran spektrum menyimulasikan pergeseran bin.",
                f"MSE citra {kurva.iloc[0, 1]:.3f} menjadi {kurva.iloc[-1, 1]:.3f}. MSE spektrum {kurva.iloc[0, 2]:.3f} menjadi {kurva.iloc[-1, 2]:.3f}.",
            ],
            [distorsi], [("t3_distorsi.png", 250)],
        ),
        (
            "Tugas 3 | K-means, DEC dan analisis",
            [
                f"Silhouette terbaik pada k={pilihan.loc[pilihan.silhouette.idxmax(), 'k']}. Stabilitas terbaik pada k={pilihan.loc[pilihan.stabilitas.idxmax(), 'k']}. Gabungan silhouette + 0,10 x stabilitas memilih k={k}.",
                "Citra saja kehilangan posisi puncak spektral. Spektrum saja kehilangan pola spasial. Tabel membandingkan ketiganya pada ruang fusi yang sama.",
                f"Silhouette K-means pada ruang awal {sil_awal:.3f}, partisi DEC pada ruang awal {sil_dec_awal}, dan partisi akhir pada ruangnya sendiri {sil_final:.3f}. ARI antarmetode {ari:.3f}.",
                f"DEC mengisi {jumlah_dec} klaster. Metode akhir {metode}. Pasangan yang paling sering muncul pada 15% margin terendah adalah {sulit.tolist()}. Puncak median spektrum kelompok terisi berada di bin {puncak}.",
                "Sampel dengan margin rendah berada di antara dua pilihan pusat menurut DEC. Bandingkan prototipe untuk melihat kemiripan pola tekstur dan puncak selain puncak dominan. ARI terhadap tipe bahan sebenarnya dihitung dengan label tersembunyi.",
            ],
            [pilihan.nlargest(4, "gabungan"), ablasi],
            [("t3_tsne.png", 150), ("t3_prototipe.png", 100)],
        ),
    ]

## Sel 4 · Laporan bagian dan pemeriksaan

Fungsi laporan menyusun tabel pencarian jumlah klaster, perbandingan modalitas, serta grafik dari eksekusi yang sama. PDF adalah bagian Tugas 3 untuk laporan kelompok. Verifikasi memeriksa skema `id_sampel,cluster`, cakupan ID, jumlah kelompok, aturan pemilihan jumlah klaster, dan panjang bagian PDF.

In [ ]:
from xml.sax.saxutils import escape
from zipfile import ZIP_DEFLATED, ZipFile

from pypdf import PdfReader
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.platypus import Image, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle


def laporan_tugas3(k, bagian):
    """Bangun bagian Tugas 3 dari tabel dan grafik hasil eksekusi."""
    prediksi = pd.read_csv(HASIL / "cluster.csv")
    cari = pd.read_csv(HASIL / "t3_pilihan_k.csv")
    data, klaster = bagian
    distorsi = data[2][0].copy()
    distorsi.columns = ["Mikroskop", "Sampel", "Kecerahan", "Frekuensi dominan", "Rentang baseline",
                       "Sampel spike", "Bin puncak"]
    distorsi["Sampel spike"] = distorsi["Sampel spike"].map(lambda v: f"{v:.1%}")
    modalitas = klaster[2][1].copy()
    modalitas.columns = ["Masukan", "Silhouette ruang sendiri", "Silhouette ruang fusi", "ARI vs fusi"]
    ukuran = prediksi.cluster.value_counts().sort_index()
    ukuran_df = pd.DataFrame(
        [(int(i), int(n), f"{n / len(prediksi):.1%}") for i, n in ukuran.items()],
        columns=["Klaster", "Sampel", "Proporsi"],
    )
    kandidat = cari.rename(columns={"silhouette":"Silhouette", "stabilitas":"Stabilitas ARI",
                                    "gabungan":"Gabungan"})
    pasangan = pd.read_csv(HASIL / "t3_pembanding.csv")
    memakai_dec = np.array_equal(prediksi.cluster, pasangan.DEC)
    metode = "DEC" if memakai_dec else "K-means pada embedding gabungan"

    navy = colors.HexColor("#173452")
    teal = colors.HexColor("#166254")
    muted = colors.HexColor("#506477")
    gaya = getSampleStyleSheet()
    gaya.add(ParagraphStyle("label_t3", fontName="Helvetica-Bold", fontSize=9, leading=12,
                             textColor=teal, spaceAfter=7))
    gaya.add(ParagraphStyle("judul_t3", fontName="Helvetica-Bold", fontSize=21, leading=25,
                             textColor=navy, spaceAfter=10))
    gaya.add(ParagraphStyle("isi_t3", fontName="Helvetica", fontSize=9.6, leading=13.5,
                             textColor=navy, spaceAfter=7))
    gaya.add(ParagraphStyle("catatan_t3", parent=gaya["isi_t3"], fontSize=8.4, leading=11, textColor=muted))
    gaya.add(ParagraphStyle("th_t3", parent=gaya["isi_t3"], fontName="Helvetica-Bold", fontSize=8.4,
                             leading=10.5, textColor=colors.white, spaceAfter=0))
    gaya.add(ParagraphStyle("td_t3", parent=gaya["isi_t3"], fontSize=8.4, leading=10.5, spaceAfter=0))
    alur = []

    def p(teks, jenis="isi_t3"):
        alur.append(Paragraph(escape(teks), gaya[jenis]))

    def judul(label, teks):
        p(label, "label_t3")
        p(teks, "judul_t3")

    def tabel(frame, lebar, sorot=None, padding=4):
        def format_nilai(x):
            return f"{x:.3f}" if isinstance(x, (float, np.floating)) else str(x)
        rows = [[Paragraph(escape(str(v)), gaya["th_t3"]) for v in frame.columns]]
        rows.extend([
            [Paragraph(escape(format_nilai(v)), gaya["td_t3"]) for v in row]
            for row in frame.itertuples(index=False, name=None)
        ])
        t = Table(rows, colWidths=lebar, repeatRows=1, hAlign="LEFT")
        aturan = [
            ("BACKGROUND", (0, 0), (-1, 0), navy),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.HexColor("#EDF4F7"), colors.white]),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("TOPPADDING", (0, 0), (-1, -1), padding),
            ("BOTTOMPADDING", (0, 0), (-1, -1), padding),
        ]
        if sorot is not None:
            aturan.append(("BACKGROUND", (0, sorot), (-1, sorot), colors.HexColor("#DCEFE7")))
        t.setStyle(TableStyle(aturan))
        alur.extend([t, Spacer(1, 9)])

    def figur(nama, tinggi, keterangan):
        im = Image(str(HASIL / nama))
        faktor = min(490/im.imageWidth, tinggi/im.imageHeight)
        im.drawWidth, im.drawHeight = im.imageWidth*faktor, im.imageHeight*faktor
        alur.append(im)
        p(keterangan, "catatan_t3")

    judul("TUGAS 3 / EKSPLORASI", "Klaster material multimodal")
    p(f"Eksperimen mengelompokkan {len(prediksi):,} sampel berdasarkan citra mikrostruktur dan spektrum. "
      f"Pencarian memilih {k} klaster. Berkas pengumpulan memakai partisi {metode}.")
    p(data[1][0])
    tabel(distorsi, [73, 58, 68, 77, 78, 69, 67])
    figur("t3_distorsi.png", 315,
          "Contoh citra dan spektrum per mikroskop. Histogram memperlihatkan orientasi Fourier dan "
          "frekuensi tekstur dominan. Spektrum bersih dinormalisasi sehingga skala vertikalnya berbeda dari spektrum mentah.")
    p("Kecerahan dinormalisasi per citra. Daya Fourier diringkas menurut jarak dari pusat dan magnitudo "
      "harmonik agar representasi kurang sensitif terhadap rotasi. Deskriptor frekuensi merangkum perubahan skala tekstur.")
    p(data[1][1])
    p("Kolom spike menghitung proporsi sampel dengan setidaknya satu bin yang menyimpang lebih dari 0,55 "
      "terhadap median lokal. Perbedaan bin puncak dapat berasal dari perubahan alat atau perbedaan komposisi sampel.",
      "catatan_t3")

    alur.append(PageBreak())
    judul("TUGAS 3 / REPRESENTASI DAN PEMILIHAN", "Embedding dan jumlah klaster")
    p("Autoencoder citra berukuran laten 10 dan autoencoder spektrum berukuran laten 14. Keduanya dilatih "
      "18 epoch untuk merekonstruksi fitur bersih. Citra diberi noise dan pergeseran satu bin radial sebagai proksi skala. "
      "Spektrum diberi noise dan pergeseran sampai tiga bin.")
    p(data[1][3])
    p("Embedding distandardisasi per dimensi dan dibagi akar ukuran laten. Cabang citra diberi bobot tetap 0,70 "
      "sebelum konkatenasi. K-means dilatih pada setiap modalitas serta pada ruang gabungan.")
    k_sil = int(cari.loc[cari.silhouette.idxmax(), "k"])
    stabil = np.isclose(cari.stabilitas, cari.stabilitas.max(), atol=1e-9, rtol=0)
    k_stab = ", ".join(str(int(x)) for x in cari.loc[stabil, "k"])
    p(f"Silhouette tertinggi pada k={k_sil}. Stabilitas tertinggi pada k={k_stab}. "
      f"Gabungan silhouette + 0,10 x stabilitas memilih k={k}.")
    tabel(kandidat, [55, 145, 145, 145], sorot=int(cari.index[cari.k == k][0]) + 1, padding=2.5)
    p("Stabilitas adalah ARI antara partisi dua resampling pada sampel penilaian yang sama. "
      "Baris berwarna menunjukkan jumlah klaster yang digunakan.", "catatan_t3")
    tabel(modalitas, [98, 138, 138, 116])
    bersama = modalitas.set_index("Masukan")["Silhouette ruang fusi"]
    p("Citra menyumbang struktur spasial. Spektrum menyumbang posisi dan intensitas puncak. "
      f"Di ruang fusi yang sama, silhouette gabungan berubah {bersama['Fusi']-bersama['Citra']:+.3f} "
      f"dibanding citra dan {bersama['Fusi']-bersama['Spektrum']:+.3f} dibanding spektrum. "
      "Spektrum sudah memisahkan sebagian besar kelompok yang ditemukan model fusi.")

    alur.append(PageBreak())
    judul("TUGAS 3 / EVALUASI", "Partisi akhir dan sampel perbatasan")
    p("DEC memulai dari pusat K-means, lalu memperbarui encoder dan pusat berdasarkan probabilitas Student-t. "
      "Optimisasi menggabungkan KL divergence dan penalti rekonstruksi 0,30 untuk menjaga informasi masukan.")
    p(klaster[1][2])
    p(klaster[1][3])
    rendah = pasangan.margin_DEC <= pasangan.margin_DEC.quantile(.15)
    berbeda = pasangan.KMeans != pasangan.DEC
    p("Margin adalah selisih dua probabilitas penugasan terbesar. Pada 15% sampel dengan margin terendah, "
      f"{berbeda[rendah].mean():.2%} berpindah klaster dari K-means ke DEC, dibanding "
      f"{berbeda[~rendah].mean():.2%} pada sampel lainnya. Kedekatan dua skor penugasan membuat sampel "
      "perbatasan lebih sensitif terhadap perubahan embedding.")
    figur("t3_tsne.png", 158, "t-SNE dari sampel embedding akhir. Panel kanan menggunakan margin DEC.")
    figur("t3_prototipe.png", 103, "Prototipe pasangan pada margin terendah. Kurva adalah median spektrum bersih.")
    if len(ukuran_df) <= 6:
        tabel(ukuran_df, [110, 190, 190], padding=2.5)
    else:
        kelompok = ", ".join(f"{i} ({n})" for i, n in ukuran.items())
        p(f"Ukuran kelompok akhir: {kelompok}.")
    p("ARI di laporan mengukur kesepakatan partisi atau stabilitas. Penilaian ARI terhadap material sebenarnya "
      "memerlukan label referensi tersembunyi.", "catatan_t3")
    p("Reproduksi: jalankan_tugas3() pada notebook Klaster_Material.ipynb. "
      f"Seed pelatihan 17. PyTorch {torch.__version__}, NumPy {np.__version__}, scikit-learn {sklearn.__version__}. "
      f"Perangkat {DEVICE}.", "catatan_t3")
    p("Referensi: Xie, Girshick, dan Farhadi (2016), Unsupervised Deep Embedding for Clustering Analysis, "
      "PMLR 48. van der Maaten dan Hinton (2008), Visualizing Data using t-SNE, JMLR 9.", "catatan_t3")

    def footer(canvas, doc):
        canvas.setFont("Helvetica", 8)
        canvas.setFillColor(muted)
        canvas.setStrokeColor(colors.HexColor("#D3E1E8"))
        canvas.line(47, 42, 548, 42)
        canvas.drawString(47, 28, "UTS DEEP LEARNING  /  KLASTER MATERIAL")
        canvas.drawRightString(548, 28, f"T3 / {doc.page}")

    path = HASIL / "03_klaster_material.pdf"
    pdf = SimpleDocTemplate(str(path), pagesize=A4, title="Tugas 3 - Klaster Material",
                            leftMargin=44, rightMargin=44, topMargin=39, bottomMargin=52)
    pdf.build(alur, onFirstPage=footer, onLaterPages=footer)
    return path


def verifikasi_tugas3(k, laporan):
    meta = pd.read_csv(DATA / "clustering_meta.csv", usecols=["id_sampel"])
    keluaran = pd.read_csv(HASIL / "cluster.csv")
    assert list(keluaran.columns) == ["id_sampel", "cluster"]
    assert keluaran.id_sampel.is_unique and not keluaran.isna().any().any()
    assert set(keluaran.id_sampel) == set(meta.id_sampel)
    assert pd.api.types.is_integer_dtype(keluaran.cluster)
    assert keluaran.cluster.nunique() == k
    catatan = pd.read_csv(HASIL / "t3_pilihan_k.csv")
    np.testing.assert_allclose(catatan.gabungan, catatan.silhouette + .10*catatan.stabilitas)
    assert int(catatan.loc[catatan.gabungan.idxmax(), "k"]) == k
    halaman = len(PdfReader(laporan).pages)
    assert 1 <= halaman <= 5
    hasil = {
        "status": "LULUS", "sampel": len(keluaran), "klaster": k, "halaman_bagian": halaman,
        "ukuran_klaster": {int(i): int(v) for i, v in keluaran.cluster.value_counts().sort_index().items()},
        "versi": {"torch": str(torch.__version__), "numpy": np.__version__, "pandas": pd.__version__,
                  "scipy": scipy.__version__, "scikit-learn": sklearn.__version__},
    }
    (HASIL / "verifikasi_tugas3.json").write_text(json.dumps(hasil, indent=2), encoding="utf-8")
    print(json.dumps(hasil, indent=2))
    return hasil


def jalankan_tugas3():
    with threadpool_limits(limits=4):
        k, bagian = tugas3()
        laporan = laporan_tugas3(k, bagian)
        return verifikasi_tugas3(k, laporan)

## Sel 5 · Eksekusi Tugas 3

Satu pemanggilan `jalankan_tugas3()` membaca data, melatih semua model, menghasilkan `cluster.csv`, membuat PDF bagian, dan mencetak status verifikasi. Lanjut ke ekspor setelah statusnya `LULUS`.

In [ ]:
hasil_tugas3 = jalankan_tugas3()

## Sel 6 · Ekspor hasil

Arsip menyusun `cluster.csv` di root dan PDF bagian di `laporan_bagian/`. Tabel, grafik, dan catatan verifikasi tersedia di `bukti_eksperimen/tugas3/`. Simpan PDF bagian bersama bagian Tugas 1 dan 2 untuk integrasi. Berkas pengumpulan gabungan memakai nama `laporan.pdf`.

In [ ]:
wajib = ["cluster.csv", "03_klaster_material.pdf", "verifikasi_tugas3.json"]
for nama in wajib:
    if not (HASIL / nama).is_file():
        raise FileNotFoundError(f"Jalankan Sel 5 terlebih dahulu: {nama}")

arsip = HASIL.parent / "hasil_tugas3.zip"
with ZipFile(arsip, "w", compression=ZIP_DEFLATED) as zip_hasil:
    zip_hasil.write(HASIL / "cluster.csv", "cluster.csv")
    zip_hasil.write(HASIL / "03_klaster_material.pdf", "laporan_bagian/03_klaster_material.pdf")
    for sumber in sorted(HASIL.iterdir()):
        if sumber.is_file() and sumber.name not in {"cluster.csv", "03_klaster_material.pdf"}:
            zip_hasil.write(sumber, f"bukti_eksperimen/tugas3/{sumber.name}")

print("Arsip:", arsip)
if drive is not None:
    from google.colab import files
    files.download(str(arsip))